# Match Filtered NIST MOFs to CSD Entries by DOI

**Inputs:**
- `data/mof_adsorbents_for_cif_matching.json` — filtered MOFs from notebook 01
- `data/step-02.csv` — scraped CSD MOF subset from notebook 02

**Output:**
- `data/matched_mofs.csv` — alphabetically sorted matched MOFs with CSD identifiers

In [1]:
import pandas as pd
import json

with open('data/mof_adsorbents_for_cif_matching.json', 'r', encoding='utf-8') as f:
    nist_mofs = json.load(f)

csd = pd.read_csv('data/step-02.csv', encoding='utf-8')

print(f"Filtered NIST MOFs: {len(nist_mofs)}")
print(f"CSD MOF entries:    {len(csd)}")

Filtered NIST MOFs: 469
CSD MOF entries:    135253


## Build DOI Lookups

In [2]:
# NIST side: DOI -> [MOF names]
nist_doi_to_mofs = {}
for mof in nist_mofs:
    for doi in mof.get('DOIs', []):
        doi_lower = doi.lower().strip()
        nist_doi_to_mofs.setdefault(doi_lower, []).append(mof['name'])

# CSD MOF subset side: DOI -> [CSD identifiers]
csd_doi_to_ids = {}
for _, row in csd.iterrows():
    if row['note'] != '-':
        continue
    doi = row.get('publication_doi')
    if pd.isna(doi):
        continue
    doi_lower = str(doi).lower().strip()
    csd_doi_to_ids.setdefault(doi_lower, []).append(row['identifier'])

shared_dois = set(nist_doi_to_mofs.keys()) & set(csd_doi_to_ids.keys())

print(f"NIST unique DOIs:  {len(nist_doi_to_mofs)}")
print(f"CSD unique DOIs:   {len(csd_doi_to_ids)}")
print(f"Overlapping DOIs:  {len(shared_dois)}")

NIST unique DOIs:  358
CSD unique DOIs:   55764
Overlapping DOIs:  201


## Match by DOI (MOF subset)

In [3]:
doi_matches = []

for doi in shared_dois:
    nist_names = list(set(nist_doi_to_mofs[doi]))
    csd_ids = csd_doi_to_ids[doi]
    
    for nist_name in nist_names:
        hashkey = next((m['hashkey'] for m in nist_mofs if m['name'] == nist_name), None)
        for csd_id in csd_ids:
            doi_matches.append({
                'nist_name': nist_name,
                'nist_hashkey': hashkey,
                'csd_identifier': csd_id,
                'doi': doi,
                'source': 'MOF subset',
            })

df_matched = pd.DataFrame(doi_matches)
matched_names_subset = set(df_matched['nist_name'].unique()) if len(df_matched) > 0 else set()

print(f"Matched from MOF subset: {len(matched_names_subset)} unique NIST MOFs")
print(f"Total match rows:        {len(df_matched)}")

Matched from MOF subset: 247 unique NIST MOFs
Total match rows:        469


## Fallback: Search the Full CSD for Unmatched DOIs

Some MOFs may not be in the CSD MOF subset but could still have crystal structures 
elsewhere in the full CSD. This cell uses the CSD Python API to search the entire 
database for entries matching unmatched DOIs.

**Requires the CSD Python API kernel.**

In [4]:
# Collect unmatched MOFs and their DOIs
unmatched_mofs = [m for m in nist_mofs if m['name'] not in matched_names_subset]

unmatched_dois = set()
unmatched_doi_to_mofs = {}
for mof in unmatched_mofs:
    for doi in mof.get('DOIs', []):
        doi_lower = doi.lower().strip()
        unmatched_dois.add(doi_lower)
        unmatched_doi_to_mofs.setdefault(doi_lower, []).append(mof['name'])

print(f"Unmatched NIST MOFs: {len(unmatched_mofs)}")
print(f"DOIs to search in full CSD: {len(unmatched_dois)}")

Unmatched NIST MOFs: 222
DOIs to search in full CSD: 149


In [5]:
import ccdc.io
from tqdm import tqdm

# Build a DOI -> [identifier] map from the FULL CSD (not just MOF subset)
# This loops through every entry in the CSD — takes 25 minutes

reader = ccdc.io.EntryReader('CSD')
print(f"Total CSD entries: {len(reader)}")

full_csd_matches = []
found_count = 0

for entry in tqdm(reader, total=len(reader), desc="Scanning full CSD"):
    doi = entry.publication.doi
    if doi is None:
        continue
    doi_lower = doi.lower().strip()
    if doi_lower in unmatched_dois:
        for nist_name in set(unmatched_doi_to_mofs[doi_lower]):
            hashkey = next((m['hashkey'] for m in nist_mofs if m['name'] == nist_name), None)
            full_csd_matches.append({
                'nist_name': nist_name,
                'nist_hashkey': hashkey,
                'csd_identifier': entry.identifier,
                'doi': doi_lower,
                'source': 'full CSD',
            })

df_full = pd.DataFrame(full_csd_matches)
new_names = set(df_full['nist_name'].unique()) - matched_names_subset if len(df_full) > 0 else set()
print(f"\nAdditional MOFs found in full CSD: {len(new_names)}")
print(f"Additional match rows: {len(df_full)}")

Total CSD entries: 1413222


Scanning full CSD: 100%|██████████| 1413222/1413222 [24:07<00:00, 976.61it/s] 


Additional MOFs found in full CSD: 14
Additional match rows: 27


## Combine & Export

In [6]:
# Combine MOF subset + full CSD matches
df_all = pd.concat([df_matched, df_full], ignore_index=True)

# Add CSD metadata where available (from step-02.csv)
csd_meta = csd[['identifier', 'formula', 'has_disorder']].drop_duplicates(subset='identifier', keep='first')
csd_meta = csd_meta.rename(columns={'identifier': 'csd_identifier'})
df_all = df_all.merge(csd_meta, on='csd_identifier', how='left')

# Sort alphabetically by NIST name, then by CSD identifier
df_all = df_all.sort_values(['nist_name', 'csd_identifier']).reset_index(drop=True)

# Save
df_all.to_csv('data/matched_mofs.csv', index=False, encoding='utf-8')

total = len(nist_mofs)
matched = df_all['nist_name'].nunique()

print("=" * 50)
print("MATCHING SUMMARY")
print("=" * 50)
print(f"Starting NIST MOFs:           {total}")
print(f"Matched (MOF subset):         {len(matched_names_subset)}")
print(f"Matched (full CSD fallback):  {len(new_names)}")
print(f"Total matched:                {matched} ({100*matched/total:.1f}%)")
print(f"Still unmatched:              {total - matched}")
print(f"\nSaved: data/matched_mofs.csv ({len(df_all)} rows)")

MATCHING SUMMARY
Starting NIST MOFs:           469
Matched (MOF subset):         247
Matched (full CSD fallback):  14
Total matched:                261 (55.7%)
Still unmatched:              208

Saved: data/matched_mofs.csv (496 rows)


In [7]:
# Save unmatched for reference
all_matched_names = set(df_all['nist_name'].unique())
still_unmatched = [m for m in nist_mofs if m['name'] not in all_matched_names]
pd.DataFrame(still_unmatched).to_csv('data/unmatched_mofs.csv', index=False, encoding='utf-8')
print(f"Saved: data/unmatched_mofs.csv ({len(still_unmatched)} MOFs)")

Saved: data/unmatched_mofs.csv (208 MOFs)


In [8]:
df_all.head(15)

,nist_name,nist_hashkey,csd_identifier,doi,source,formula,has_disorder
0,((Me2NH2)In(NH2BDC)2),NIST-MATDB-895aa514507d242d5062b7a2742cea92,XALROT,10.1039/c2cc16923a,MOF subset,(C16 H10 In1 N2 O8)n,True
1,(Et2NH2)3[(Cu4Cl)3(TTCA)8],NIST-MATDB-698d87ea3dc222af53aca3746f3227d3,REGYOT,10.1039/c2cc35461f,MOF subset,"(C168 H72 Cl3 Cu12 O48 3-)n,26n(C5 H11 N1 O1),...",True
2,(H3O)4[Ni6(pi3-O)2(pi2-OSC2H6)2(SO4)2(TATB)8/3...,NIST-MATDB-ee36b2a9095ec91288836a44edfdc823,UGODEB,10.1021/acs.inorgchem.5b00316,MOF subset,"(C102 H66 N12 Ni9 O42 S6 6-)n,6n(H3 O1 1+),6n(...",True
3,(In3O)(OH)(ADC)2(IN)2*4.67H2O,NIST-MATDB-e0f1ea383b8b290ca83fb4c214ff9cf0,KACZEV,10.1039/c0cc02808h,MOF subset,"(C120 H75 In9 N18 O42)n,14n(H2 O1)",True
4,(In3O)(OH)(ADC)2(IN)2*4.67H2O,NIST-MATDB-e0f1ea383b8b290ca83fb4c214ff9cf0,KACZIZ,10.1039/c0cc02808h,MOF subset,"(C120 H81 In9 N24 O42)n,8n(H2 O1)",True
5,(Me2NH2)2(DMF)9(H2O)5,NIST-MATDB-0fc6fe0e88747d6206743f79868a947c,DEYMOL,10.1021/ic302583a,MOF subset,"(C37 H20 In2 O20 2-)n,2n(C2 H8 N1 1+),9n(C3 H7...",True
6,(NH2(CH3)2)3[Zn6(BTC)4(BTB)],NIST-MATDB-1e1291030fcd194b53599b4d457548d1,SIJDUM,10.1039/c3ce40929e,MOF subset,"(C63 H27 O30 Zn6 3-)n,3n(C2 H8 N1 1+)",True
7,(me2NH2)6[In10(TTCA)12]*24DMF*15H2O,NIST-MATDB-dc1f4b7ddd96245d3704815ee72b638d,WUBWUN,10.1021/ic501413r,full CSD,NaN,NaN
8,"2,4,6-tris-(4-carboxyphenoxy)-1,3,5-triazine (...",NIST-MATDB-9e095580c7def92493b44834013f53af,RUSCUF,10.1039/c5dt01770j,MOF subset,(C24 H13 N3 O10 Zn2)n,True
9,3Inf[(Cu(mu4-O)(mu2-OH)2(Me2trzpba)4],NIST-MATDB-eac31584ad49217cba58bde85c79de24,MALFUD,10.1016/j.micromeso.2010.11.017,MOF subset,"(C44 H42 Cu4 N12 O11)n,10n(H2 O1)",True
